# PCR — Priority-Conditioned Retention (test-time, Mamba-790m)
**Importance-Aware Capacity Allocation in a pretrained continuous-state SSM**

Training-free inference-time retention control on pretrained Mamba-790m.

- Scope: single model (790m). 1.4b failure and natural-language fragility are reported as limitations.
- Two levers: (1) survival via Δ gate, (2) precision via write quantization.
- Runs on L4 in tens of minutes. Do NOT install `mamba-ssm`/`causal-conv1d` — slow path required for Δ hooks.

In [25]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Jun 20 05:03:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   59C    P0             29W /   72W |    3472MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:
# block fast path → force slow path so dt_proj/conv1d hooks fire
!pip -q install "transformers>=4.39" accelerate
import torch, random, math
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

torch 2.11.0+cu128 | cuda True


In [27]:
# load pretrained 790m, fp32, frozen
from transformers import AutoTokenizer, MambaForCausalLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "state-spaces/mamba-790m-hf"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = MambaForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)
NL = len(model.backbone.layers)
print("layers", NL)
print("note: 'fast path is not available ... sequential implementation' is expected — slow path confirmed")

Loading weights:   0%|          | 0/482 [00:00<?, ?it/s]

layers 48
note: 'fast path is not available ... sequential implementation' is expected — slow path confirmed


In [28]:
# per-position gain on dt_proj output (pre-softplus); controlled via _CTRL
_CTRL = {'gain_vec': None, 'layers': None}
def _mk_dt_hook(idx):
    def hook(mod, inp, out):
        g = _CTRL['gain_vec']
        if g is None: return out
        if _CTRL['layers'] is not None and idx not in _CTRL['layers']: return out
        L = out.shape[1]
        gv = g[:L].to(out.dtype).to(out.device)
        return out * gv.view(1, L, 1)
    return hook
_dt_handles = []
def setup_dt_hooks():
    global _dt_handles
    for h in _dt_handles: h.remove()
    _dt_handles = [l.mixer.dt_proj.register_forward_hook(_mk_dt_hook(i))
                   for i, l in enumerate(model.backbone.layers)]
def verify_dt_hooks():
    ids = tokenizer(" a b c d e f", return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad(): base = model(ids).logits[0, -1].clone()
    _CTRL['gain_vec'] = torch.full((ids.shape[1],), 0.5); _CTRL['layers'] = None
    with torch.no_grad(): pert = model(ids).logits[0, -1]
    _CTRL['gain_vec'] = None
    d = (base - pert).abs().max().item()
    return d > 1e-4, d
setup_dt_hooks()
ok, d = verify_dt_hooks(); print("Δ hook fires:", ok, "| max logit change", round(d, 3))

Δ hook fires: True | max logit change 106.302


In [29]:
# mean decay A (-exp(A_log)) per layer; A < 0 means recency bias
_A = [(-torch.exp(l.mixer.A_log)).mean().item() for l in model.backbone.layers]
print("mean A: L0", round(_A[0],3), "| Lmid", round(_A[NL//2],3), "| Llast", round(_A[-1],3))

mean A: L0 -33.752 | Lmid -42.991 | Llast -25.894


In [30]:
# single-token word pool + MQAR builder (each word = 1 token → clean position accounting)
def single_token_words(n, seed=0):
    rng = random.Random(seed); ids = list(range(1000, 45000)); rng.shuffle(ids); out = []
    for i in ids:
        s = tokenizer.decode([i]).strip()
        if (s.isascii() and s.isalpha() and 3 <= len(s) <= 8
                and len(tokenizer(" " + s, add_special_tokens=False).input_ids) == 1):
            out.append(s)
        if len(out) >= n: break
    return out
_WORDS = single_token_words(4000); print("single-token words:", len(_WORDS))

def _enc(word):
    return tokenizer(" " + word, add_special_tokens=False).input_ids

def build_mqar(N, rng, query_pair=0):
    ws = rng.sample(_WORDS, 2 * N)
    pairs = [(ws[2*j], ws[2*j+1]) for j in range(N)]
    seq = []; pos = []
    for (k, v) in pairs:
        kp = len(seq); seq += _enc(k)
        vp = len(seq); seq += _enc(v); pos.append((kp, vp))
    seq += tokenizer(" ?", add_special_tokens=False).input_ids
    seq += _enc(pairs[query_pair][0])
    ids = torch.tensor([seq], device=DEVICE)
    tgt = _enc(pairs[query_pair][1])[0]
    return ids, pos, tgt, pairs

def _recall_once(ids, tgt):
    with torch.no_grad(): lg = model(ids).logits[0, -1]
    return int(lg.argmax().item() == tgt)

single-token words: 4000


In [31]:
# capacity curve: find N where base recall ≈ 0.4
def base_recall(N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t)); h += _recall_once(ids, tgt)
    return h / n
print("[capacity curve]")
for N in [2, 8, 16, 24, 32, 48, 64]:
    print(f"  N={N:>3}: {base_recall(N, n=30):.2f}")

[capacity curve]
  N=  2: 0.63
  N=  8: 0.77
  N= 16: 0.77
  N= 24: 0.63
  N= 32: 0.53
  N= 48: 0.43
  N= 64: 0.33


In [32]:
# Gate1: downstream Δ suppression (og < 1) → target retention
def trials_delta(N, og, layers=None, n=30, seed0=0, scope='downstream'):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        L = ids.shape[1]; v0 = pos[0][1]; gv = torch.ones(L)
        if scope == 'downstream':
            gv[v0 + 1:] = og
        elif scope == 'random':
            idxs = list(range(v0 + 1, L)); random.Random(seed0 + t + 999).shuffle(idxs)
            for j in idxs[:max(1, len(idxs)//2)]: gv[j] = og
        _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
        h += _recall_once(ids, tgt)
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return h / n

N = 32
b = base_recall(N); print("base", round(b, 2))
print("[Gate1 dose] og sweep (all layers)")
for og in [1.0, 0.9, 0.8, 0.7, 0.6, 0.4]:
    print(f"  og={og}: {trials_delta(N, og, n=30):.2f}")
print("[Gate1 specificity] downstream vs random (og=0.8)")
print(f"  downstream {trials_delta(N,0.8,scope='downstream',n=30):.2f} | random {trials_delta(N,0.8,scope='random',n=30):.2f}")

base 0.53
[Gate1 dose] og sweep (all layers)
  og=1.0: 0.53
  og=0.9: 0.60
  og=0.8: 0.73
  og=0.7: 0.60
  og=0.6: 0.30
  og=0.4: 0.00
[Gate1 specificity] downstream vs random (og=0.8)
  downstream 0.73 | random 0.50


In [33]:
# layer localization: which 8-layer band carries the recall gain?
groups = {f"L{a}-{a+7}": list(range(a, a + 8)) for a in range(0, NL, 8)}
base = base_recall(N)
print(f"[layer localization] og=0.8, base={base:.2f}")
for name, g in groups.items():
    r = trials_delta(N, 0.8, layers=g, n=30)
    print(f"  {name}: {r:.2f}  (Δ {r-base:+.2f})")

[layer localization] og=0.8, base=0.53
  L0-7: 0.53  (Δ +0.00)
  L8-15: 0.53  (Δ +0.00)
  L16-23: 0.67  (Δ +0.13)
  L24-31: 0.60  (Δ +0.07)
  L32-39: 0.53  (Δ +0.00)
  L40-47: 0.53  (Δ +0.00)


In [34]:
# surgical L16-23 vs all-layer + ppl guardrail
def ppl_under_gain(og, layers=None):
    txt = ("The transformer relies on attention while state space models compress "
           "history into a fixed hidden state for linear time inference.")
    ids = tokenizer(txt, return_tensors='pt').input_ids.to(DEVICE); L = ids.shape[1]
    _CTRL['gain_vec'] = torch.full((L,), og); _CTRL['layers'] = set(layers) if layers else None
    with torch.no_grad(): nll = model(ids, labels=ids).loss.item()
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return math.exp(nll)
base_ppl = ppl_under_gain(1.0); L1623 = list(range(16, 24))
print(f"base ppl {base_ppl:.1f}")
print(f"  full   og=0.8: recall {trials_delta(N,0.8,n=30):.2f}  ppl x{ppl_under_gain(0.8)/base_ppl:.2f}")
print(f"  L16-23 og=0.8: recall {trials_delta(N,0.8,layers=L1623,n=30):.2f}  ppl x{ppl_under_gain(0.8,L1623)/base_ppl:.2f}")

base ppl 232.2
  full   og=0.8: recall 0.73  ppl x1.81
  L16-23 og=0.8: recall 0.67  ppl x1.07


In [35]:
# write-order as implicit priority: recall by pair position
def recall_of_pair(N, j, og, layers=None, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, pairs = build_mqar(N, random.Random(seed0 + t), query_pair=j)
        L = ids.shape[1]; gv = torch.ones(L)
        if og < 1.0: gv[pos[j][1] + 1:] = og
        _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
        h += _recall_once(ids, tgt)
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return h / n
print("[write-order priority] N=32, L16-23 og=0.8")
for j, lab in [(0, 'early(P1)'), (N//2, 'mid(P2)'), (N-1, 'late(P3)')]:
    bb = recall_of_pair(N, j, 1.0, n=30); aa = recall_of_pair(N, j, 0.8, layers=L1623, n=30)
    print(f"  {lab:>10} pair{j:>2}: base {bb:.2f} -> {aa:.2f}  (Δ {aa-bb:+.2f})")

[write-order priority] N=32, L16-23 og=0.8
   early(P1) pair 0: base 0.53 -> 0.67  (Δ +0.13)
     mid(P2) pair16: base 0.00 -> 0.10  (Δ +0.10)
    late(P3) pair31: base 0.03 -> 0.13  (Δ +0.10)


In [36]:
# graded P1/P2/P3: assign per-level og to each write position (exploratory)
def graded_recall(N, og_lv, layers=None, n=30, seed0=0):
    res = {1: 0, 2: 0, 3: 0}; cnt = {1: 0, 2: 0, 3: 0}
    for t in range(n):
        rng = random.Random(seed0 + t); ws = rng.sample(_WORDS, 2 * N)
        pairs = [(ws[2*i], ws[2*i+1]) for i in range(N)]
        lvl = [(i % 3) + 1 for i in range(N)]
        seq = []; pos = []
        for (k, v) in pairs:
            kp = len(seq); seq += _enc(k)
            vp = len(seq); seq += _enc(v); pos.append((kp, vp))
        for Lq in [1, 2, 3]:
            j = lvl.index(Lq)
            s2 = seq + tokenizer(" ?", add_special_tokens=False).input_ids + _enc(pairs[j][0])
            ids = torch.tensor([s2], device=DEVICE); tgt = _enc(pairs[j][1])[0]
            gv = torch.ones(ids.shape[1])
            for i, (kp, vp) in enumerate(pos):
                gv[vp] = og_lv[lvl[i]]
            _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers) if layers else None
            res[Lq] += _recall_once(ids, tgt); cnt[Lq] += 1
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return {k: res[k] / cnt[k] for k in [1, 2, 3]}
g = graded_recall(N, {1: 0.7, 2: 0.85, 3: 1.0}, layers=L1623, n=30)
print("[graded P1/P2/P3] L16-23")
print(f"  P1 {g[1]:.2f} | P2 {g[2]:.2f} | P3 {g[3]:.2f}")
print("  monotone P1>P2>P3: graded allocation holds; broken: SSM uniform-decay limit")

[graded P1/P2/P3] L16-23
  P1 0.47 | P2 0.13 | P3 0.07
  monotone P1>P2>P3: graded allocation holds; broken: SSM uniform-decay limit


In [37]:
# lever 2: write precision — fake-quantize conv1d output at selected positions
def fake_quant_vec(v, bits):
    if bits >= 16: return v
    qmax = (2 ** bits) - 1; lo = v.amin(-1, keepdim=True); hi = v.amax(-1, keepdim=True)
    scale = (hi - lo).clamp(min=1e-8) / qmax
    return torch.round((v - lo) / scale) * scale + lo
_QMASK = {'pos': None, 'bits': 16, 'fired': 0}
def _qhook(mod, inp, out):
    if _QMASK['bits'] >= 16 or not _QMASK['pos']: return out
    L = out.shape[-1]; pos = [p for p in _QMASK['pos'] if 0 <= p < L]
    if not pos: return out
    o = out.clone()
    for p in pos: o[:, :, p] = fake_quant_vec(o[:, :, p], _QMASK['bits'])
    _QMASK['fired'] += 1; return o
_qh = [l.mixer.conv1d.register_forward_hook(_qhook) for l in model.backbone.layers]

def rd_target(bits, N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        _QMASK['pos'] = list(pos[0]); _QMASK['bits'] = bits; h += _recall_once(ids, tgt)
    _QMASK['pos'] = None; _QMASK['bits'] = 16; return h / n
def rd_distractor(bits, N, n=30, seed0=0):
    h = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar(N, random.Random(seed0 + t))
        dp = [p for pr in pos[1:] for p in pr]
        _QMASK['pos'] = dp; _QMASK['bits'] = bits; h += _recall_once(ids, tgt)
    _QMASK['pos'] = None; _QMASK['bits'] = 16; return h / n
print("[A] target k-bit → recall (rate-distortion curve)")
for bt in [16, 8, 4, 2, 1]: print(f"  {bt:>2}-bit: {rd_target(bt, N, n=30):.2f}")
print("[B] distractor k-bit → target recall (priority reallocation)")
for bt in [16, 4, 2, 1]: print(f"  {bt:>2}-bit: {rd_distractor(bt, N, n=30):.2f}")

[A] target k-bit → recall (rate-distortion curve)
  16-bit: 0.53
   8-bit: 0.53
   4-bit: 0.03
   2-bit: 0.00
   1-bit: 0.00
[B] distractor k-bit → target recall (priority reallocation)
  16-bit: 0.53
   4-bit: 0.07
   2-bit: 0.00
   1-bit: 0.00


In [38]:
# exp A-1: 8-layer band localization across N=16/32/64
# does L16-23 remain the best band under varying capacity pressure?

sweep_Ns   = [16, 32, 64]
band_names = [f'L{a}-{a+7}' for a in range(0, NL, 8)]
band_layers = [list(range(a, a + 8)) for a in range(0, NL, 8)]

print('[exp A-1] band localization across N  (og=0.8)')
header = f"{'N':>4} | {'base':>5} | " + ' | '.join(f'{b:>9}' for b in band_names)
print(header)
print('-' * len(header))

nsweep_loc = {}
for N in sweep_Ns:
    n_trials = 20 if N == 64 else 30
    b = base_recall(N, n=n_trials)
    band_r = [trials_delta(N, 0.8, layers=bl, n=n_trials) for bl in band_layers]
    nsweep_loc[N] = {'base': b, 'bands': dict(zip(band_names, band_r))}
    cols = ' | '.join(f'{r:.2f}({r-b:+.2f})' for r in band_r)
    print(f'{N:>4} | {b:>5.2f} | {cols}')

print('\nbest band per N:')
for N, res in nsweep_loc.items():
    best = max(res['bands'], key=res['bands'].get)
    print(f'  N={N:>2}: {best}  recall={res["bands"][best]:.2f}  gain={res["bands"][best]-res["base"]:+.2f}')

[exp A-1] band localization across N  (og=0.8)
   N |  base |      L0-7 |     L8-15 |    L16-23 |    L24-31 |    L32-39 |    L40-47
------------------------------------------------------------------------------------
  16 |  0.77 | 0.73(-0.03) | 0.70(-0.07) | 0.83(+0.07) | 0.80(+0.03) | 0.80(+0.03) | 0.77(+0.00)
  32 |  0.53 | 0.53(+0.00) | 0.53(+0.00) | 0.67(+0.13) | 0.60(+0.07) | 0.53(+0.00) | 0.53(+0.00)
  64 |  0.40 | 0.40(+0.00) | 0.20(-0.20) | 0.60(+0.20) | 0.45(+0.05) | 0.15(-0.25) | 0.40(+0.00)

best band per N:
  N=16: L16-23  recall=0.83  gain=+0.07
  N=32: L16-23  recall=0.67  gain=+0.13
  N=64: L16-23  recall=0.60  gain=+0.20


In [39]:
# exp A-2: surgical L16-23 vs full — recall across N

print('[exp A-2] surgical (L16-23) vs full  (og=0.8)')
print(f"{'N':>4} | {'base':>6} | {'full':>6} | {'Δfull':>6} | {'surg':>6} | {'Δsurg':>6}")
print('-' * 48)

for N in sweep_Ns:
    n_trials = 20 if N == 64 else 30
    base_r = base_recall(N, n=n_trials)
    full_r  = trials_delta(N, 0.8, n=n_trials)
    surg_r  = trials_delta(N, 0.8, layers=L1623, n=n_trials)
    print(f'{N:>4} | {base_r:>6.2f} | {full_r:>6.2f} | {full_r-base_r:>+6.2f} | '
          f'{surg_r:>6.2f} | {surg_r-base_r:>+6.2f}')

_base_ppl = ppl_under_gain(1.0)
_full_ppl = ppl_under_gain(0.8)
_surg_ppl = ppl_under_gain(0.8, L1623)
print(f'\nPPL (fixed passage, N-independent): '
      f'base={_base_ppl:.1f}  '
      f'full=×{_full_ppl/_base_ppl:.2f}  '
      f'surg(L16-23)=×{_surg_ppl/_base_ppl:.2f}')

[exp A-2] surgical (L16-23) vs full  (og=0.8)
   N |   base |   full |  Δfull |   surg |  Δsurg
------------------------------------------------
  16 |   0.77 |   0.90 |  +0.13 |   0.83 |  +0.07
  32 |   0.53 |   0.73 |  +0.20 |   0.67 |  +0.13
  64 |   0.40 |   0.55 |  +0.15 |   0.60 |  +0.20

PPL (fixed passage, N-independent): base=232.2  full=×1.81  surg(L16-23)=×1.07


In [42]:
# exp B-1: fine-grained ablation within L16-23
# sliding 4-layer windows + individual layers to find minimal effective circuit

N = 32; og = 0.8; n_trials = 30

# 4-layer sliding windows inside L16-23
sub4_names  = [f'L{a}-{a+3}' for a in range(16, 24)]
sub4_layers = [list(range(a, a + 4)) for a in range(16, 24)]

# individual layers L16-L23
indiv_names  = [f'L{i}' for i in range(16, 24)]
indiv_layers = [[i] for i in range(16, 24)]

base_r = base_recall(N, n=n_trials)
L1623_r = trials_delta(N, og, layers=list(range(16, 24)), n=n_trials)
print(f'[exp B-1] fine-grained ablation within L16-23  (N={N}, og={og})')
print(f'reference — base: {base_r:.2f}  L16-23 full: {L1623_r:.2f}  (Δ {L1623_r-base_r:+.2f})')

print('\n--- 4-layer sliding windows ---')
sub4_res = {}
for name, layers in zip(sub4_names, sub4_layers):
    r = trials_delta(N, og, layers=layers, n=n_trials)
    sub4_res[name] = r
    bar = '█' * int((r - base_r) / 0.01) if r > base_r else ''
    print(f'  {name}: {r:.2f}  (Δ {r-base_r:+.2f})  {bar}')

print('\n--- individual layers ---')
indiv_res = {}
for name, layers in zip(indiv_names, indiv_layers):
    r = trials_delta(N, og, layers=layers, n=n_trials)
    indiv_res[name] = r
    bar = '█' * int((r - base_r) / 0.01) if r > base_r else ''
    print(f'  {name}: {r:.2f}  (Δ {r-base_r:+.2f})  {bar}')

best_sub4   = max(sub4_res,   key=sub4_res.get)
best_indiv  = max(indiv_res,  key=indiv_res.get)
print(f'\nbest 4-layer window: {best_sub4}  recall={sub4_res[best_sub4]:.2f}')
print(f'best single layer:   {best_indiv}  recall={indiv_res[best_indiv]:.2f}')

[exp B-1] fine-grained ablation within L16-23  (N=32, og=0.8)
reference — base: 0.53  L16-23 full: 0.67  (Δ +0.13)

--- 4-layer sliding windows ---
  L16-19: 0.63  (Δ +0.10)  █████████
  L17-20: 0.67  (Δ +0.13)  █████████████
  L18-21: 0.67  (Δ +0.13)  █████████████
  L19-22: 0.70  (Δ +0.17)  ████████████████
  L20-23: 0.60  (Δ +0.07)  ██████
  L21-24: 0.60  (Δ +0.07)  ██████
  L22-25: 0.60  (Δ +0.07)  ██████
  L23-26: 0.43  (Δ -0.10)  

--- individual layers ---
  L16: 0.53  (Δ +0.00)  
  L17: 0.57  (Δ +0.03)  ███
  L18: 0.57  (Δ +0.03)  ███
  L19: 0.60  (Δ +0.07)  ██████
  L20: 0.53  (Δ +0.00)  
  L21: 0.57  (Δ +0.03)  ███
  L22: 0.63  (Δ +0.10)  █████████
  L23: 0.40  (Δ -0.13)  

best 4-layer window: L19-22  recall=0.70
best single layer:   L22  recall=0.63


In [43]:
# exp B-2: minimal circuit — best 4-layer window vs full L16-23, with PPL
# confirms whether the 8-layer band is strictly needed or a 4-layer sub-band suffices

best_sub4_layers = sub4_layers[sub4_names.index(best_sub4)]

base_ppl_b = ppl_under_gain(1.0)
best4_ppl  = ppl_under_gain(og, best_sub4_layers)
full_ppl   = ppl_under_gain(og, list(range(16, 24)))

best4_recall = trials_delta(N, og, layers=best_sub4_layers, n=n_trials)
full_recall  = trials_delta(N, og, layers=list(range(16, 24)), n=n_trials)

print(f'[exp B-2] minimal circuit comparison  (N={N}, og={og})')
print(f"{'Intervention':<20} {'Recall':>7} {'Δ recall':>9} {'PPL mult':>9}")
print('-' * 48)
print(f"{'base':<20} {base_r:>7.2f} {'':>9} {'×1.00':>9}")
print(f"{'L16-23 (8 layers)':<20} {full_recall:>7.2f} {full_recall-base_r:>+9.2f} {f'×{full_ppl/base_ppl_b:.2f}':>9}")
print(f"{best_sub4+' (4 layers)':<20} {best4_recall:>7.2f} {best4_recall-base_r:>+9.2f} {f'×{best4_ppl/base_ppl_b:.2f}':>9}")

recall_loss = full_recall - best4_recall
ppl_saved   = full_ppl/base_ppl_b - best4_ppl/base_ppl_b
print(f'\n4-layer sub-band vs 8-layer full band:')
print(f'  recall loss: {recall_loss:+.2f}  PPL improvement: {ppl_saved:+.2f}×')
if recall_loss <= 0.03:
    print(f'  verdict: {best_sub4} is an effective minimal circuit (recall drop ≤ 0.03)')
else:
    print(f'  verdict: 8-layer band needed — {best_sub4} alone drops recall by {recall_loss:.2f}')

[exp B-2] minimal circuit comparison  (N=32, og=0.8)
Intervention          Recall  Δ recall  PPL mult
------------------------------------------------
base                    0.53               ×1.00
L16-23 (8 layers)       0.67     +0.13     ×1.07
L19-22 (4 layers)       0.70     +0.17     ×1.08

4-layer sub-band vs 8-layer full band:
  recall loss: -0.03  PPL improvement: -0.01×
  verdict: L19-22 is an effective minimal circuit (recall drop ≤ 0.03)


## Notes

- **Capacity curve**: use N where base ≈ 0.4 as the operating point (N=32 from cell 7 onward).
- **Gate1**: recall peaks in a specific og range; downstream ≫ random confirms specificity.
- **Localization (Exp A)**: L16-23 is the best band at N=16/32/64. At N=64 surgical (0.60) > full (0.55); off-target layers turn harmful under pressure.
- **Surgical (Exp B)**: 4-layer sliding windows and individual layers within L16-23 identify the minimal effective sub-circuit. Run B-2 output to see if a 4-layer band suffices or the full 8 layers are needed.
- **Write-order**: early-pair Δ gain > late-pair → write order acts as implicit priority.
- **Graded P1/P2/P3**: monotone = graded allocation holds; non-monotone = SSM uniform-decay limit (both worth reporting).
- **Write quantization**: target-quant cliff at 4-bit; distractor-quant fails — negative result, superimposed state has no per-slot addressing.

**Limitations**: effects are robust on 790m synthetic MQAR but not replicated on 1.4b and fragile on natural language.

In [44]:
# exp C-1: scale check — Mamba-130m  (24 layers, same tokenizer)
# Q: does a recall circuit exist at proportional depth in a smaller model?
from transformers import AutoTokenizer, MambaForCausalLM

MODEL_130 = "state-spaces/mamba-130m-hf"
tok130 = AutoTokenizer.from_pretrained(MODEL_130)
mod130 = MambaForCausalLM.from_pretrained(MODEL_130, torch_dtype=torch.float32).to(DEVICE).eval()
for p in mod130.parameters(): p.requires_grad_(False)
NL130 = len(mod130.backbone.layers)
print(f"Mamba-130m: {NL130} layers")

# verify tokenizer compatibility (same GPT-NeoX vocab → _WORDS pool reusable)
sample_ok = all(len(tok130(" " + w, add_special_tokens=False).input_ids) == 1
                for w in _WORDS[:200])
print(f"130m tokenizer compatible with _WORDS: {sample_ok}")

# 130m dt_proj hooks (same gain_vec structure as 790m)
_CTRL_130 = {'gain_vec': None, 'layers': None}

def _mk_dt_hook_130(idx):
    def hook(mod, inp, out):
        g = _CTRL_130['gain_vec']
        if g is None: return out
        if _CTRL_130['layers'] is not None and idx not in _CTRL_130['layers']: return out
        L = out.shape[1]
        return out * g[:L].to(out.dtype).to(out.device).view(1, L, 1)
    return hook

_dt_handles_130 = [l.mixer.dt_proj.register_forward_hook(_mk_dt_hook_130(i))
                   for i, l in enumerate(mod130.backbone.layers)]

def _verify_130():
    ids = tok130(" a b c d e f", return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad(): base = mod130(ids).logits[0, -1].clone()
    _CTRL_130['gain_vec'] = torch.full((ids.shape[1],), 0.5)
    with torch.no_grad(): pert = mod130(ids).logits[0, -1]
    _CTRL_130['gain_vec'] = None
    d = (base - pert).abs().max().item()
    return d > 1e-4, round(d, 3)
ok130, d130 = _verify_130()
print(f"130m Δ hook fires: {ok130} | max logit change {d130}")

# MQAR helpers for 130m (same protocol; _WORDS reused — identical GPT-NeoX vocab)
def _enc130(word):
    return tok130(" " + word, add_special_tokens=False).input_ids

def build_mqar_130(N, rng, query_pair=0):
    ws = rng.sample(_WORDS, 2 * N); pairs = [(ws[2*j], ws[2*j+1]) for j in range(N)]
    seq = []; pos = []
    for k, v in pairs:
        kp = len(seq); seq += _enc130(k)
        vp = len(seq); seq += _enc130(v); pos.append((kp, vp))
    seq += tok130(" ?", add_special_tokens=False).input_ids + _enc130(pairs[query_pair][0])
    ids = torch.tensor([seq], device=DEVICE)
    return ids, pos, _enc130(pairs[query_pair][1])[0], pairs

def trials_delta_130(N, og, layers=None, n=30, seed0=0):
    hits = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar_130(N, random.Random(seed0 + t))
        L = ids.shape[1]; v0 = pos[0][1]
        gv = torch.ones(L); gv[v0 + 1:] = og
        _CTRL_130['gain_vec'] = gv
        _CTRL_130['layers'] = set(layers) if layers else None
        with torch.no_grad(): hits += (mod130(ids).logits[0, -1].argmax().item() == tgt)
    _CTRL_130['gain_vec'] = None; _CTRL_130['layers'] = None
    return round(hits / n, 2)

# 4-layer window sweep  (4/24 == 8/48 — same proportional coverage as 790m sweep)
N = 32; og = 0.8; n_trials = 30
base_r130 = trials_delta_130(N, 1.0, n=n_trials)

win4_names  = [f'L{a}-{a+3}' for a in range(NL130 - 3)]
win4_layers = [list(range(a, a + 4)) for a in range(NL130 - 3)]
win4_res = {}
for name, lays in zip(win4_names, win4_layers):
    win4_res[name] = trials_delta_130(N, og, layers=lays, n=n_trials)

best_win130 = max(win4_res, key=win4_res.get)
print(f"\n[exp C-1] Mamba-130m 4-layer window sweep  (N={N}, og={og})")
print(f"base recall: {base_r130}")
for name, r in win4_res.items():
    delta = r - base_r130
    bar = '█' * max(0, int(abs(delta) * 100)) if delta >= 0 else '▒' * max(0, int(abs(delta) * 100))
    mk = '  ← BEST' if name == best_win130 else ''
    print(f"  {name:<9}  {r:.2f}  (Δ {delta:+.2f})  {bar}{mk}")

# proportional-depth prediction from 790m L19-22 (layers 19-22 out of 0-47)
prop_lo = round(19 / 47 * (NL130 - 1))
prop_hi  = round(22 / 47 * (NL130 - 1))
print(f"\nbest 4-layer window: {best_win130}  recall={win4_res[best_win130]}")
print(f"790m proportional prediction: L{prop_lo}–L{prop_hi}  ({prop_lo/(NL130-1):.0%}–{prop_hi/(NL130-1):.0%} depth)")

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.79k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/517M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Mamba-130m: 24 layers
130m tokenizer compatible with _WORDS: True
130m Δ hook fires: True | max logit change 40.9

[exp C-1] Mamba-130m 4-layer window sweep  (N=32, og=0.8)
base recall: 0.57
  L0-3       0.53  (Δ -0.04)  ▒▒▒
  L1-4       0.57  (Δ +0.00)  
  L2-5       0.57  (Δ +0.00)  
  L3-6       0.57  (Δ +0.00)  
  L4-7       0.53  (Δ -0.04)  ▒▒▒
  L5-8       0.53  (Δ -0.04)  ▒▒▒
  L6-9       0.57  (Δ +0.00)  
  L7-10      0.57  (Δ +0.00)  
  L8-11      0.57  (Δ +0.00)  
  L9-12      0.47  (Δ -0.10)  ▒▒▒▒▒▒▒▒▒
  L10-13     0.50  (Δ -0.07)  ▒▒▒▒▒▒
  L11-14     0.53  (Δ -0.04)  ▒▒▒
  L12-15     0.43  (Δ -0.14)  ▒▒▒▒▒▒▒▒▒▒▒▒▒
  L13-16     0.63  (Δ +0.06)  ██████  ← BEST
  L14-17     0.40  (Δ -0.17)  ▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
  L15-18     0.33  (Δ -0.24)  ▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
  L16-19     0.23  (Δ -0.34)  ▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
  L17-20     0.20  (Δ -0.37)  ▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
  L18-21     0.43  (Δ -0.14)  ▒▒▒▒▒▒▒▒▒▒▒▒▒
  L19-22     0.47  (Δ -0.10)  ▒▒▒▒▒▒▒▒▒
  L2

In [45]:
# exp C-2: 130m minimal circuit — 2-layer sub-windows + PPL + scale comparison
def ppl_under_gain_130(og, layers=None):
    txt = ("The transformer relies on attention while state space models compress "
           "history into a fixed hidden state for linear time inference.")
    ids = tok130(txt, return_tensors='pt').input_ids.to(DEVICE)
    _CTRL_130['gain_vec'] = torch.full((ids.shape[1],), og)
    _CTRL_130['layers'] = set(layers) if layers else None
    with torch.no_grad(): nll = mod130(ids, labels=ids).loss.item()
    _CTRL_130['gain_vec'] = None; _CTRL_130['layers'] = None
    return math.exp(nll)

best_win130_layers = win4_layers[win4_names.index(best_win130)]

# 2-layer sub-windows within best 4-layer window  (proportional to 790m B-1 individual layers)
sub2_names  = [f'L{a}-{a+1}' for a in best_win130_layers[:-1]]
sub2_layers = [[a, a + 1] for a in best_win130_layers[:-1]]
sub2_res = {n: trials_delta_130(N, og, layers=l, n=n_trials)
            for n, l in zip(sub2_names, sub2_layers)}
best_sub2_130 = max(sub2_res, key=sub2_res.get)

# individual layers within best window
indiv_res130 = {f'L{l}': trials_delta_130(N, og, layers=[l], n=n_trials)
                for l in best_win130_layers}

print(f"[exp C-2] fine-grained ablation within {best_win130}  (N={N}, og={og})")
print("--- 2-layer sub-windows ---")
for name, r in sub2_res.items():
    mk = '  ← BEST' if name == best_sub2_130 else ''
    print(f"  {name}: {r:.2f}  (Δ {r - base_r130:+.2f}){mk}")
print("--- individual layers ---")
for name, r in indiv_res130.items():
    print(f"  {name}: {r:.2f}  (Δ {r - base_r130:+.2f})")

# PPL guardrail
base_ppl130  = ppl_under_gain_130(1.0)
best4_ppl130 = ppl_under_gain_130(og, best_win130_layers)
full_r130    = trials_delta_130(N, og, layers=list(range(NL130)), n=n_trials)
full_ppl130  = ppl_under_gain_130(og, list(range(NL130)))

print(f"\n[exp C-2] recall + PPL — Mamba-130m")
print(f"{'Intervention':<28}  Recall  Δ recall  PPL mult")
print(f"{'base':<28}  {base_r130:.2f}             ×1.00")
print(f"{best_win130+' (4L)':<28}  {win4_res[best_win130]:.2f}  "
      f"    {win4_res[best_win130]-base_r130:+.2f}     ×{best4_ppl130/base_ppl130:.2f}")
print(f"{'all '+str(NL130)+' layers':<28}  {full_r130:.2f}             ×{full_ppl130/base_ppl130:.2f}")

# scale comparison
lo_idx = int(best_win130.split('-')[0][1:])
hi_idx = lo_idx + 3
depth_lo = round(lo_idx / (NL130 - 1) * 100)
depth_hi = round(hi_idx / (NL130 - 1) * 100)
print(f"\n[scale comparison — N=32, og=0.8]")
print(f"{'Model':<8}  {'Layers':>6}  {'Circuit':<13}  {'Depth':>10}  Recall  Δ recall  PPL mult")
print(f"{'790m':<8}  {'48':>6}  {'L19-22 (4L)':<13}  {'40%–47%':>10}  0.70    +0.17     ×1.08")
print(f"{'130m':<8}  {str(NL130):>6}  {best_win130+' (4L)':<13}  "
      f"  {depth_lo}%–{depth_hi}%    {win4_res[best_win130]:.2f}    "
      f"{win4_res[best_win130]-base_r130:+.2f}      ×{best4_ppl130/base_ppl130:.2f}")
prop_lo = round(19 / 47 * (NL130 - 1)); prop_hi = round(22 / 47 * (NL130 - 1))
match = best_win130 == f'L{prop_lo}-{prop_hi}'
print(f"\nproportion-depth match (predicted L{prop_lo}-{prop_hi}): {match}")

[exp C-2] fine-grained ablation within L13-16  (N=32, og=0.8)
--- 2-layer sub-windows ---
  L13-14: 0.57  (Δ +0.00)
  L14-15: 0.60  (Δ +0.03)  ← BEST
  L15-16: 0.57  (Δ +0.00)
--- individual layers ---
  L13: 0.57  (Δ +0.00)
  L14: 0.57  (Δ +0.00)
  L15: 0.57  (Δ +0.00)
  L16: 0.57  (Δ +0.00)

[exp C-2] recall + PPL — Mamba-130m
Intervention                  Recall  Δ recall  PPL mult
base                          0.57             ×1.00
L13-16 (4L)                   0.63      +0.06     ×0.95
all 24 layers                 0.43             ×1.31

[scale comparison — N=32, og=0.8]
Model     Layers  Circuit             Depth  Recall  Δ recall  PPL mult
790m          48  L19-22 (4L)       40%–47%  0.70    +0.17     ×1.08
130m          24  L13-16 (4L)      57%–70%    0.63    +0.06      ×0.95

proportion-depth match (predicted L9-11): False


In [46]:
# exp D-1: JRT (Just Read Twice) baseline — doubled context, no weight change
# Arora et al. (2024): repeat the k-v context so the SSM compresses it a second time
# Compare: base / JRT / PCR (L19-22) / JRT+PCR across N=16/32/64

def build_mqar_jrt(N, rng, query_pair=0):
    """Repeat the k-v pairs before the query (read twice)."""
    ws = rng.sample(_WORDS, 2 * N); pairs = [(ws[2*j], ws[2*j+1]) for j in range(N)]
    seq = []; pos = []
    for k, v in pairs:
        kp = len(seq); seq += _enc(k)
        vp = len(seq); seq += _enc(v); pos.append((kp, vp))
    for k, v in pairs:           # second read
        seq += _enc(k); seq += _enc(v)
    seq += tokenizer(" ?", add_special_tokens=False).input_ids + _enc(pairs[query_pair][0])
    ids = torch.tensor([seq], device=DEVICE)
    return ids, pos, _enc(pairs[query_pair][1])[0], pairs

def recall_jrt(N, n=30, seed0=0):
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    hits = 0
    for t in range(n):
        ids, _, tgt, _ = build_mqar_jrt(N, random.Random(seed0 + t))
        with torch.no_grad(): hits += _recall_once(ids, tgt)
    return round(hits / n, 2)

def recall_jrt_pcr(N, og, layers, n=30, seed0=0):
    """JRT context + PCR gain on target's downstream positions."""
    hits = 0
    for t in range(n):
        ids, pos, tgt, _ = build_mqar_jrt(N, random.Random(seed0 + t))
        L = ids.shape[1]; v0 = pos[0][1]
        gv = torch.ones(L); gv[v0 + 1:] = og
        _CTRL['gain_vec'] = gv; _CTRL['layers'] = set(layers)
        with torch.no_grad(): hits += _recall_once(ids, tgt)
    _CTRL['gain_vec'] = None; _CTRL['layers'] = None
    return round(hits / n, 2)

pcr_layers = list(range(19, 23))   # L19-22 minimal circuit
sweep_Ns = [16, 32, 64]

# sequence length overhead (N-dependent)
ids_b, _, _, _ = build_mqar(32, random.Random(0))
ids_j, _, _, _ = build_mqar_jrt(32, random.Random(0))
print(f"sequence length at N=32: base={ids_b.shape[1]}  JRT={ids_j.shape[1]}  "
      f"overhead=×{ids_j.shape[1]/ids_b.shape[1]:.2f}")

print(f"\n[exp D-1] JRT vs PCR vs base  (og=0.8, L19-22)")
print(f"{'N':>4}  {'base':>6}  {'JRT':>6}  {'PCR':>6}  {'JRT+PCR':>7}")
print('-' * 38)
d1_res = {}
for N in sweep_Ns:
    nt = 20 if N == 64 else 30
    b   = base_recall(N, n=nt)
    jrt = recall_jrt(N, n=nt)
    pcr = trials_delta(N, 0.8, layers=pcr_layers, n=nt)
    jp  = recall_jrt_pcr(N, 0.8, pcr_layers, n=nt)
    d1_res[N] = {'base': b, 'jrt': jrt, 'pcr': pcr, 'jrt_pcr': jp}
    print(f'{N:>4}  {b:>6.2f}  {jrt:>6.2f}  {pcr:>6.2f}  {jp:>7.2f}')

sequence length at N=32: base=66  JRT=130  overhead=×1.97

[exp D-1] JRT vs PCR vs base  (og=0.8, L19-22)
   N    base     JRT     PCR  JRT+PCR
--------------------------------------
  16    0.77    1.00    0.87     1.00
  32    0.53    1.00    0.70     1.00
  64    0.40    0.95    0.35     0.95


In [47]:
# exp D-2: method comparison summary at N=32 operating point
N = 32; og = 0.8; nt = 30

b32  = d1_res[32]['base']
jrt32 = d1_res[32]['jrt']
pcr32 = d1_res[32]['pcr']
jp32  = d1_res[32]['jrt_pcr']

ids_b32, _, _, _ = build_mqar(N, random.Random(0))
ids_j32, _, _, _ = build_mqar_jrt(N, random.Random(0))
L_base = ids_b32.shape[1]; L_jrt = ids_j32.shape[1]

print(f'[exp D-2] method comparison at N={N}, og={og}')
print(f"{'Method':<18}  {'Recall':>6}  {'Δ recall':>8}  {'Seq len':>8}  {'Seq overhead':>12}")
print('-' * 62)
print(f"{'base':<18}  {b32:>6.2f}  {'---':>8}  {L_base:>8}  {'×1.00':>12}")
print(f"{'JRT':<18}  {jrt32:>6.2f}  {jrt32-b32:>+8.2f}  {L_jrt:>8}  {f'×{L_jrt/L_base:.2f}':>12}")
print(f"{'PCR (L19-22)':<18}  {pcr32:>6.2f}  {pcr32-b32:>+8.2f}  {L_base:>8}  {'×1.00':>12}")
print(f"{'JRT + PCR':<18}  {jp32:>6.2f}  {jp32-b32:>+8.2f}  {L_jrt:>8}  {f'×{L_jrt/L_base:.2f}':>12}")

print(f'\nPCR vs JRT: recall delta {pcr32-jrt32:+.2f}  seq overhead PCR saves ×{L_jrt/L_base:.2f}')
additive = abs((jp32 - b32) - ((jrt32 - b32) + (pcr32 - b32))) < 0.05
print(f'JRT+PCR gains additive: {additive}  '
      f'(sum of individual gains={jrt32-b32+pcr32-b32:+.2f}, actual={jp32-b32:+.2f})')

[exp D-2] method comparison at N=32, og=0.8
Method              Recall  Δ recall   Seq len  Seq overhead
--------------------------------------------------------------
base                  0.53       ---        66         ×1.00
JRT                   1.00     +0.47       130         ×1.97
PCR (L19-22)          0.70     +0.17        66         ×1.00
JRT + PCR             1.00     +0.47       130         ×1.97

PCR vs JRT: recall delta -0.30  seq overhead PCR saves ×1.97
JRT+PCR gains additive: False  (sum of individual gains=+0.63, actual=+0.47)
